# 1. Import Libraries

In [16]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pulp

# 2. Read Data Files 

In [17]:
bom = pd.read_csv('data/bill_of_materials.csv')
cakes = pd.read_csv('data/cakes.csv')
channels = pd.read_csv('data/channels.csv')
ingredients = pd.read_csv('data/ingredients.csv')
demand_params = pd.read_csv('data/instructor_demand_competition.csv')
wages_energy = pd.read_csv('data/wages_energy.csv')

# 3. Extract Wages and Cost Parameters

In [19]:
param_map = dict(zip(wages_energy.parameter, wages_energy.value))
prep_wage = param_map['prep_wage_usd_per_hour']
oven_wage = param_map['oven_wage_usd_per_hour']
pack_wage = param_map['pack_wage_usd_per_hour']
oven_rental = param_map['oven_rental_usd_per_hour']
oven_cost = param_map['oven_cost_usd_per_hour']
budget = param_map['budget_usd']

print(f"Preparation Wage: ${prep_wage}/hour")
print(f"Oven Wage: ${oven_wage}/hour")
print(f"Packaging Wage: ${pack_wage}/hour")
print(f"Oven Rental Cost: ${oven_rental}/hour")
print(f"Oven Operating Cost: ${oven_cost}/hour")
print(f"Total Budget: ${budget}")

Preparation Wage: $4.0/hour
Oven Wage: $5.0/hour
Packaging Wage: $3.5/hour
Oven Rental Cost: $3.0/hour
Oven Operating Cost: $1.4/hour
Total Budget: $4000.0


In [21]:
cake_info = cakes.set_index('cake_id')
channel_info = channels.set_index('channel')
ingredient_cost = ingredients.set_index('ingredient')['unit_cost_usd'].to_dict()
print(cake_info)
print(channel_info)
print(ingredient_cost)

                                name  batch_size_units  oven_min_per_batch  \
cake_id                                                                      
1        Triple Chocolate Layer Cake                 6                  50   
2                    Victoria Sponge                 8                  39   
3                    Red Velvet Cake                 6                  48   
4                 Lemon Drizzle Cake                 8                  73   
5                New York Cheesecake                 4                  97   
6           Flourless Chocolate Cake                 8                  42   
7                        Banana Cake                 8                  68   
8          Old-Fashioned Marble Cake                 6                  80   
9                       Coconut Cake                 6                  46   
10              Coffee & Walnut Cake                 8                  46   

         prep_min_per_unit  pack_min_per_unit  packaging_cost_p

In [ ]:
ingredient_cols = [c for c in bom.columns if c not in ['cake_id','name']]
usage = bom.set_index('cake_id')[ingredient_cols]

print(usage)
print(ingredient_cols)

         Banana  Butter  Chocolate  CocoaPowder  CoconutFlakes  CreamCheese  \
cake_id                                                                       
1           0.0   0.000       0.10         0.06           0.00          0.0   
2           0.0   0.200       0.00         0.00           0.00          0.0   
3           0.0   0.000       0.00         0.02           0.00          0.3   
4           0.0   0.225       0.00         0.00           0.00          0.0   
5           0.0   0.060       0.00         0.00           0.00          0.9   
6           0.0   0.170       0.35         0.00           0.00          0.0   
7           0.3   0.120       0.00         0.00           0.00          0.0   
8           0.0   0.250       0.00         0.03           0.00          0.0   
9           0.0   0.170       0.00         0.00           0.16          0.0   
10          0.0   0.225       0.00         0.00           0.00          0.0   

         Dairy  Eggs_each  Flavorings  Flour  Leave

# Objective function (maximize profit)

Profit = Revenue - IngredientCost - LaborCost - UtilitiesCost - TransportCost - PackagingCost

price: dict (cake_id, channel) -> price value
sales_vars: dict (cake_id, channel) -> pulp variable s_ij
ingredient_inv_vars: dict ingredient -> purchased quantity (inv_k)
prep_hours_var, pack_hours_var, oven_hours_var: capacity hour decision varstransport_cost: dict channel -> cost per unit
packaging_cost_per_unit: dict cake_id -> packaging cost per unit produced
cakes, channels: dataframes with structural info
ingredient_cost: dict ingredient -> unit cost
oven_rental, oven_cost: per hour costs
*_wage: wage rates per hour

In [ ]:
def objective_function(model, price, sales_vars, ingredient_inv_vars, prep_hours_var, pack_hours_var, oven_hours_var, transport_cost, packaging_cost_per_unit, cakes, channels, ingredient_cost, oven_rental, oven_cost, prep_wage, pack_wage, oven_wage):
   
    revenue = pulp.lpSum(price[(i,j)] * sales_vars[(i,j)] for (i,j) in sales_vars)
    ingredient_cost_total = pulp.lpSum(ingredient_cost[k] * ingredient_inv_vars[k] for k in ingredient_inv_vars)
    labor_cost = prep_hours_var * prep_wage + pack_hours_var * pack_wage + oven_hours_var * oven_wage
    utilities_cost = oven_hours_var * (oven_rental + oven_cost)
    transport_cost_total = pulp.lpSum(transport_cost[j] * sales_vars[(i,j)] for (i,j) in sales_vars)
    packaging_cost_total = pulp.lpSum(packaging_cost_per_unit[i] * sales_vars[(i,j)] for (i,j) in sales_vars)  # assume packaging consumed per unit sold
    model += revenue - ingredient_cost_total - labor_cost - utilities_cost - transport_cost_total - packaging_cost_total, 'Total_Profit'
    return model

In [ ]:
def build_profit_max_model(price_df):
    """Build linear ILP with prices treated as parameters.

    price_df columns: cake_id, channel, price
    Returns: model, variable dictionaries for inspection.
    """
    # Prepare price dict and demand dict
    price = {(int(row.cake_id), row.channel): float(row.price) for row in price_df.itertuples()}
    # Merge demand parameters with prices to compute linear demand D_ij = alpha - beta * P_ij
    demand_df = demand_params.merge(price_df, left_on=['ID','channel'], right_on=['cake_id','channel'], how='left')
    demand_df['cake_id'] = demand_df['ID']
    demand_df['D_ij'] = demand_df['alpha'] - demand_df['beta'] * demand_df['price']
    # Clip negative to zero
    demand_df.loc[demand_df['D_ij'] < 0, 'D_ij'] = 0
    demand = {(int(r.cake_id), r.channel): float(r.D_ij) for r in demand_df.itertuples()}
    # Initialize model
    model = pulp.LpProblem('SweetMarketProfitMax', pulp.LpMaximize)
    cake_ids = cakes.cake_id.tolist()
    channel_list = channels.channel.tolist()
    # Decision variables
    y = {(i,j): pulp.LpVariable(f'y_{i}_{j}', lowBound=0, cat='Integer') for i in cake_ids for j in channel_list}  # production units for channel j
    s = {(i,j): pulp.LpVariable(f's_{i}_{j}', lowBound=0, cat='Integer') for i in cake_ids for j in channel_list}  # sales units
    b = {i: pulp.LpVariable(f'b_{i}', lowBound=0, cat='Integer') for i in cake_ids}  # batches
    delta = {i: pulp.LpVariable(f'delta_{i}', lowBound=0, upBound=1, cat='Binary') for i in cake_ids}  # indicates cake produced (for min units logic)
    # Ingredient inventory purchase variables (ensure feasibility)
    ing_vars = {k: pulp.LpVariable(f'inv_{k}', lowBound=0) for k in ingredient_cols}  # continuous quantity purchased
    # Capacity hour decision variables (investment)
    prep_hours = pulp.LpVariable('prep_hours', lowBound=0)
    pack_hours = pulp.LpVariable('pack_hours', lowBound=0)
    oven_hours = pulp.LpVariable('oven_hours', lowBound=0)
    # Transport and packaging cost maps
    transport_cost = channel_info.transport_cost_per_unit_usd.to_dict()
    packaging_cost_per_unit = cake_info.packaging_cost_per_unit_usd.to_dict()
    # OBJECTIVE FUNCTION (attach after components defined)
    objective_function(model, price, s, ing_vars, prep_hours, pack_hours, oven_hours, transport_cost, packaging_cost_per_unit, cakes, channels, ingredient_cost, oven_rental, oven_cost, prep_wage, pack_wage, oven_wage)
    # Constraints linearizing s_ij = min(D_ij, y_ij): s_ij ≤ D_ij and s_ij ≤ y_ij
    for (i,j), var in s.items():
        model += var <= demand[(i,j)], f'demand_cap_{i}_{j}'
        model += var <= y[(i,j)], f'sales_leq_prod_{i}_{j}'
    # Batch size linking and production aggregation
    for i in cake_ids:
        batch_size = cake_info.loc[i, 'batch_size_units']
        model += pulp.lpSum(y[(i,j)] for j in channel_list) == b[i] * batch_size, f'batch_link_{i}'
    # Minimum units if made logic: if delta_i =1 then total production >= minimum_units_if_made_i; also b_i >= delta_i
    for i in cake_ids:
        min_units = cake_info.loc[i, 'minimum_units_if_made']
        model += pulp.lpSum(y[(i,j)] for j in channel_list) >= min_units * delta[i], f'min_units_{i}'
        model += b[i] >= delta[i], f'b_geq_delta_{i}'
    # Stage capacities (minutes used ≤ hours * 60)
    prep_usage = pulp.lpSum(cake_info.loc[i,'prep_min_per_unit'] * y[(i,j)] for i in cake_ids for j in channel_list)
    pack_usage = pulp.lpSum(cake_info.loc[i,'pack_min_per_unit'] * y[(i,j)] for i in cake_ids for j in channel_list)
    oven_usage = pulp.lpSum(cake_info.loc[i,'oven_min_per_batch'] * b[i] for i in cake_ids)
    model += prep_usage <= prep_hours * 60, 'prep_capacity'
    model += pack_usage <= pack_hours * 60, 'pack_capacity'
    model += oven_usage <= oven_hours * 60, 'oven_capacity'
    # Ingredient constraints: total usage ≤ purchased inventory ing_vars[k]
    for k in ingredient_cols:
        usage_k = pulp.lpSum(usage.loc[i,k] * pulp.lpSum(y[(i,j)] for j in channel_list) for i in cake_ids)
        model += usage_k <= ing_vars[k], f'ingredient_cap_{k}'
    # Channel service capacities
    for j in channel_list:
        cap_j = channel_info.loc[j,'service_cap_per_week']
        model += pulp.lpSum(s[(i,j)] for i in cake_ids) <= cap_j, f'channel_cap_{j}'
    # Budget constraint: Sum of investment/purchase costs ≤ budget
    ingredient_purchase_cost = pulp.lpSum(ingredient_cost[k] * ing_vars[k] for k in ingredient_cols)
    labor_invest_cost = prep_hours * prep_wage + pack_hours * pack_wage + oven_hours * oven_wage
    utilities_invest_cost = oven_hours * (oven_rental + oven_cost)
    model += ingredient_purchase_cost + labor_invest_cost + utilities_invest_cost <= budget, 'budget_constraint'
    # Non-negativity and integrality already enforced by variable types.
    return model, {'y': y, 's': s, 'b': b, 'delta': delta, 'ing': ing_vars, 'prep_hours': prep_hours, 'pack_hours': pack_hours, 'oven_hours': oven_hours, 'demand': demand}

In [ ]:
# Example price dataframe construction (placeholder values; replace with strategy results)
example_prices = []
for i in cakes.cake_id:
    for j in channels.channel:
        # Simple placeholder: base price = 5 + cake_id * 0.5 (adjust manually later)
        example_prices.append({'cake_id': i, 'channel': j, 'price': 5 + i*0.5})
price_df = pd.DataFrame(example_prices)
model, vars_dict = build_profit_max_model(price_df)
# Solve (can change solver)
model.solve(pulp.PULP_CBC_CMD(msg=0))
print('Status:', pulp.LpStatus[model.status])
print('Objective (Profit):', pulp.value(model.objective))
# Display sample results for first cake
first_cake = cakes.cake_id.iloc[0]
for ch in channels.channel:
    print(f'Cake {first_cake} channel {ch} produced', vars_dict['y'][(first_cake,ch)].value(), 'sold', vars_dict['s'][(first_cake,ch)].value())